Data Source: https://www.healthquality.va.gov/HEALTHQUALITY/guidelines/Pain/cot/VADODOpioidsCPG.pdf

_**Retreival Augmented Generation**_

Chunking

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader("Clinical Practice Guidelines for Use of Opioid for Chronic Pain Management.pdf")
pages = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = splitter.split_documents(pages)

Importing Embedding Model

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\iyers\anaconda3\envs\ragenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\iyers\anaconda3\envs\ragenv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Chroma as Vector Database

In [3]:
from langchain_community.vectorstores import Chroma

import shutil
shutil.rmtree("./chroma_db", ignore_errors=True)

vectorstore = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory="./chroma_db"
)

Using Llama model because it can accomodate multiple instructions.

In [4]:
%pip install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key,  
    temperature=0
)

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template("""
Use the context below to answer the question. 
At the end of your answer, cite the source by mentioning the page numbers from the guideline pdf.
Only answer what is required. 
If the answer is not in the context, say "I don't know" and do not attempt to make up an answer.
Answer concisely and accurately.
When mentioning abbreviations, write out the full form at least once.

Context: {context}
Question: {question}
Answer:""")

def format_docs_with_sources(docs):
    output = []
    for doc in docs:
        page = doc.metadata.get("page", "unknown")
        output.append(f"[Page {page}] {doc.page_content}")
    return "\n\n".join(output)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = (
    {"context": retriever | format_docs_with_sources, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [7]:
question = input("How can I help you today?")
result = qa_chain.invoke(question)
print("Answer:", result)

Answer: The Opioid Taper Decision Tool is an example of tapers for opioids offered by the VHA PBM Academic Detailing Service. 

Source: VHA PBM Academic Detailing Service guideline (page not specified in the context, but mentioned on page 30)


_**Evaluation**_

Dataset: https://github.com/mmahbub/cpgQA

In [9]:
import time
import pandas as pd
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from datasets import Dataset

df = pd.read_csv("cpgQA-v1.0.csv")
df = df.sample(5, random_state=20)

questions = df["QUESTION"].tolist()
ground_truths = df["ANSWER"].tolist()

answers = []
contexts = []

for q in questions:
    result = qa_chain.invoke(q)
    answers.append(result)
    docs = retriever.invoke(q)
    contexts.append([doc.page_content for doc in docs])
    time.sleep(2)

eval_data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(eval_data)

evaluator_llm = LangchainLLMWrapper(ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key,
    temperature=0,
    n=1
))

evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

print(results)

Evaluating:  50%|█████     | 10/20 [02:50<02:47, 16.70s/it]Exception raised in Job[7]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Evaluating:  55%|█████▌    | 11/20 [03:00<02:09, 14.38s/it]Exception raised in Job[15]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Evaluating:  90%|█████████ | 18/20 [03:01<00:07,  3.71s/it]Exception raised in Job[3]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Evaluating: 100%|██████████| 20/20 [03:02<00:00,  9.11s/it]


{'faithfulness': 0.0000, 'answer_relevancy': 0.1436, 'context_precision': 0.0000, 'context_recall': 0.3000}


In [10]:
%pip install evaluate bert-score rouge-score --quiet

Note: you may need to restart the kernel to use updated packages.


In [14]:
from evaluate import load

bertscore = load("bertscore")
bert_results = bertscore.compute(predictions=answers, references=ground_truths, lang="en")

print("BERTScore Precision:", sum(bert_results["precision"]) / len(bert_results["precision"]))
print("BERTScore Recall:", sum(bert_results["recall"]) / len(bert_results["recall"]))
print("BERTScore F1:", sum(bert_results["f1"]) / len(bert_results["f1"]))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore Precision: 0.8093299746513367
BERTScore Recall: 0.8299431443214417
BERTScore F1: 0.818957793712616
